# AI in the Charity & Nonprofit Sector — Data Cleaning Notebook
**Project:** Ready or Left Behind? AI in the Charity & Nonprofit Sector  
**Author:** Elvira  
**GitHub:** github.com/elviraqc/charity-nonprofit-ai-analysis  
**Last updated:** April 2026  

## Purpose
This notebook cleans and processes the Charity Commission full register 
(open government data) to produce summary tables for Tableau Public dashboard 
View 5 — Who Gets Left Behind.

All source data is from the Charity Commission for England and Wales open 
register, downloaded from:
register-of-charities.charitycommission.gov.uk/register/full-register-download

## Output
- `charity_commission_cleaned.xlsx` — four summary sheets for Tableau Public
- Four audit CSVs — full cleaned datasets for GitHub reference

## Step 1 — Load Raw Data
Loading all four Charity Commission TSV files. Files are large (46MB–162MB) 
so `on_bad_lines='skip'` is used to handle rows where free-text fields 
(charity descriptions) contain tab characters that break the parser.

In [25]:
import pandas as pd
import os

# Paths
BASE = r"C:\Portfolio\P1-AI-nonprofit"
RAW  = os.path.join(BASE, "data", "charity-commission-register")
OUT  = os.path.join(BASE, "data")

print("Raw data folder:", RAW)
print("Output folder:  ", OUT)

# Check files are visible
print("\nFiles in register folder:")
for f in os.listdir(RAW):
    size_mb = os.path.getsize(os.path.join(RAW, f)) / 1024 / 1024
    print(f"  {f:60s}  {size_mb:.1f} MB")

Raw data folder: C:\Portfolio\P1-AI-nonprofit\data\charity-commission-register
Output folder:   C:\Portfolio\P1-AI-nonprofit\data

Files in register folder:
  charity.txt                                                   153.0 MB
  charity_annual_return_parta.txt                               161.8 MB
  charity_area_of_operation.txt                                 46.4 MB
  charity_classification.txt                                    135.2 MB
  Data Definition.docx                                          0.1 MB


## Step 2 — Filter and Clean Charity Table
Filtering to registered charities only (removing 210,565 removed/linked 
charities). Adding income bands to enable size segmentation in Tableau.
Income bands follow standard charity sector definitions used by NCVO and 
Charity Commission reporting.

In [26]:
# Load charity table
print("Loading charity table — this may take 20-30 seconds...")

charity = pd.read_csv(
    os.path.join(RAW, "charity.txt"),
    sep="\t",
    encoding="utf-8",
    dtype=str,
    low_memory=False,
    on_bad_lines="skip"
)

print(f"Rows loaded:  {len(charity):,}")
print(f"Columns:      {list(charity.columns)}")

Loading charity table — this may take 20-30 seconds...
Rows loaded:  394,619
Columns:      ['date_of_extract', 'organisation_number', 'registered_charity_number', 'linked_charity_number', 'charity_name', 'charity_type', 'charity_registration_status', 'date_of_registration', 'date_of_removal', 'charity_reporting_status', 'latest_acc_fin_period_start_date', 'latest_acc_fin_period_end_date', 'latest_income', 'latest_expenditure', 'charity_contact_address1', 'charity_contact_address2', 'charity_contact_address3', 'charity_contact_address4', 'charity_contact_address5', 'charity_contact_postcode', 'charity_contact_phone', 'charity_contact_email', 'charity_contact_web', 'charity_company_registration_number', 'charity_insolvent', 'charity_in_administration', 'charity_previously_excepted', 'charity_is_cdf_or_cif', 'charity_is_cio', 'cio_is_dissolved', 'date_cio_dissolution_notice', 'charity_activities', 'charity_gift_aid', 'charity_has_land']


In [27]:
# Filter to registered charities only and keep relevant columns
charity_clean = charity[
    charity['charity_registration_status'] == 'Registered'
].copy()

print(f"Registered charities: {len(charity_clean):,}")
print(f"Removed/other:        {len(charity) - len(charity_clean):,}")

# Keep only columns we need
cols_keep = [
    'organisation_number',
    'registered_charity_number',
    'charity_name',
    'charity_type',
    'charity_registration_status',
    'date_of_registration',
    'latest_income',
    'latest_expenditure',
    'charity_contact_postcode'
]
charity_clean = charity_clean[cols_keep]

# Convert income and expenditure to numeric
charity_clean['latest_income'] = pd.to_numeric(
    charity_clean['latest_income'], errors='coerce')
charity_clean['latest_expenditure'] = pd.to_numeric(
    charity_clean['latest_expenditure'], errors='coerce')

# Add income band
def income_band(income):
    if pd.isna(income):
        return 'Unknown'
    elif income < 10000:
        return 'Under 10k'
    elif income < 100000:
        return '10k-100k'
    elif income < 500000:
        return '100k-500k'
    elif income < 1000000:
        return '500k-1m'
    else:
        return 'Over 1m'

charity_clean['income_band'] = charity_clean['latest_income'].apply(income_band)

# Check distribution
print("\nIncome band distribution:")
print(charity_clean['income_band'].value_counts().sort_index())
print("\nSample:")
print(charity_clean.head(3))

Registered charities: 184,054
Removed/other:        210,565

Income band distribution:
income_band
100k-500k    27553
10k-100k     59284
500k-1m       5883
Over 1m       9214
Under 10k    59652
Unknown      22468
Name: count, dtype: int64

Sample:
  organisation_number registered_charity_number  \
1                   2                    200027   
4                   5                    200034   
5                   6                    200034   

                      charity_name charity_type charity_registration_status  \
1              HITCHAM FREE CHURCH          NaN                  Registered   
4  CLOPHILL RELIEF IN NEED CHARITY          NaN                  Registered   
5                   MARL ALLOTMENT          NaN                  Registered   

          date_of_registration  latest_income  latest_expenditure  \
1  1962-05-17 00:00:00.0000000            NaN                 NaN   
4  1972-07-19 00:00:00.0000000            NaN                 NaN   
5  1961-07-07 00:00:00.

## Step 3 — Charity Type Classification
Extracting 'What' classifications only — these describe what the charity does 
(Education, Health, Poverty relief etc). Each charity can have multiple 
classifications. 17 categories in total across 644,414 classification rows.

In [28]:
# Load classification table
print("Loading classification table...")

classification = pd.read_csv(
    os.path.join(RAW, "charity_classification.txt"),
    sep="\t",
    encoding="utf-8",
    dtype=str,
    low_memory=False,
    on_bad_lines="skip"
)

print(f"Rows loaded: {len(classification):,}")
print(f"Columns:     {list(classification.columns)}")

# Keep only What classification (charity type)
what_clean = classification[
    classification['classification_type'] == 'What'
].copy()

what_clean = what_clean[[
    'organisation_number',
    'classification_type',
    'classification_description'
]]

print(f"\nWhat classification rows: {len(what_clean):,}")
print("\nTop 20 charity types:")
print(what_clean['classification_description'].value_counts().head(20))

Loading classification table...
Rows loaded: 1,732,218
Columns:     ['date_of_extract', 'organisation_number', 'registered_charity_number', 'linked_charity_number', 'classification_code', 'classification_type', 'classification_description']

What classification rows: 644,414

Top 20 charity types:
classification_description
Education/training                                                134378
General Charitable Purposes                                        87895
The Prevention Or Relief Of Poverty                                52366
Religious Activities                                               50765
The Advancement Of Health Or Saving Of Lives                       45205
Arts/culture/heritage/science                                      44657
Disability                                                         41982
Amateur Sport                                                      39743
Economic/community Development/employment                          34284
Environment/conse

## Step 4 — Geographic Reach
Extracting area of operation data. Three geographic levels available: 
Local Authority, Region, and Country. Used to understand where charities 
operate — note this is self-reported by charities and one charity can 
list multiple areas.

In [29]:
# Load area of operation table
print("Loading area of operation table...")

area = pd.read_csv(
    os.path.join(RAW, "charity_area_of_operation.txt"),
    sep="\t",
    encoding="utf-8",
    dtype=str,
    low_memory=False,
    on_bad_lines="skip"
)

print(f"Rows loaded: {len(area):,}")
print(f"Columns:     {list(area.columns)}")

# Keep relevant columns
area_clean = area[[
    'organisation_number',
    'geographic_area_type',
    'geographic_area_description',
    'parent_geographic_area_description'
]].copy()

print("\nGeographic area types:")
print(area_clean['geographic_area_type'].value_counts())

print("\nTop 20 areas of operation:")
print(area_clean['geographic_area_description'].value_counts().head(20))

Loading area of operation table...
Rows loaded: 537,048
Columns:     ['date_of_extract', 'organisation_number', 'registered_charity_number', 'linked_charity_number', 'geographic_area_type', 'geographic_area_description', 'parent_geographic_area_type', 'parent_geographic_area_description', 'welsh_ind']

Geographic area types:
geographic_area_type
Local Authority    299726
Country            168276
Region              69046
Name: count, dtype: int64

Top 20 areas of operation:
geographic_area_description
Throughout England And Wales    43011
Throughout England              13227
Throughout London                8946
Scotland                         8746
Kent                             7807
Essex                            6931
Surrey                           6848
Hampshire                        6840
Devon                            6031
Lancashire                       5653
Northern Ireland                 5623
Norfolk                          5421
North Yorkshire                  516

## Step 5 — Financial Data (Annual Return Part A)
Loading financial submissions. Contains income, expenditure, government 
funding, and volunteer counts. Charities submit annually — multiple years 
per charity so we keep the most recent year only.

In [30]:
# Load annual return part A
print("Loading annual return part A...")

parta = pd.read_csv(
    os.path.join(RAW, "charity_annual_return_parta.txt"),
    sep="\t",
    encoding="utf-8",
    dtype=str,
    low_memory=False,
    on_bad_lines="skip"
)

print(f"Rows loaded: {len(parta):,}")
print(f"Columns:     {list(parta.columns)}")

Loading annual return part A...
Rows loaded: 654,849
Columns:     ['date_of_extract', 'organisation_number', 'registered_charity_number', 'latest_fin_period_submitted_ind', 'fin_period_order_number', 'ar_cycle_reference', 'fin_period_start_date', 'fin_period_end_date', 'ar_due_date', 'ar_received_date', 'total_gross_income', 'total_gross_expenditure', 'charity_raises_funds_from_public', 'charity_professional_fundraiser', 'charity_agreement_professional_fundraiser', 'charity_commercial_participator', 'charity_agreement_commerical_participator', 'grant_making_is_main_activity', 'charity_receives_govt_funding_contracts', 'count_govt_contracts', 'charity_receives_govt_funding_grants', 'count_govt_grants', 'income_from_government_contracts', 'income_from_government_grants', 'charity_has_trading_subsidiary', 'trustee_also_director_of_subsidiary', 'does_trustee_receive_any_benefit', 'trustee_payments_acting_as_trustee', 'trustee_receives_payments_services', 'trustee_receives_other_benefit', '

## Step 6 — Clean Financials and Keep Latest Year
Keeping most recent financial period per charity only. Key fields: 
government contract income, government grant income, volunteer count. 
These are the core equity indicators for View 5.

In [31]:
# Clean annual return part A
parta_clean = parta[[
    'organisation_number',
    'fin_period_end_date',
    'total_gross_income',
    'total_gross_expenditure',
    'charity_receives_govt_funding_contracts',
    'charity_receives_govt_funding_grants',
    'income_from_government_contracts',
    'income_from_government_grants',
    'count_volunteers'
]].copy()

# Convert numeric columns
num_cols = [
    'total_gross_income',
    'total_gross_expenditure',
    'income_from_government_contracts',
    'income_from_government_grants',
    'count_volunteers'
]
for col in num_cols:
    parta_clean[col] = pd.to_numeric(parta_clean[col], errors='coerce')

# Convert date
parta_clean['fin_period_end_date'] = pd.to_datetime(
    parta_clean['fin_period_end_date'], errors='coerce'
)

# Keep most recent year only per charity
parta_latest = (
    parta_clean
    .sort_values('fin_period_end_date', ascending=False)
    .groupby('organisation_number')
    .first()
    .reset_index()
)

print(f"Charities with financial data: {len(parta_latest):,}")
print(f"\nGovernment funding summary:")
print(parta_latest['charity_receives_govt_funding_contracts'].value_counts())
print()
print(parta_latest['charity_receives_govt_funding_grants'].value_counts())

Charities with financial data: 174,263

Government funding summary:
charity_receives_govt_funding_contracts
False    113232
True       7360
Name: count, dtype: int64

charity_receives_govt_funding_grants
False    87602
True     32990
Name: count, dtype: int64


Really interesting data. 32,990 charities receive government grants and 7,360 receive government contracts — that's your funding dependency picture for View 5. Charities reliant on government funding are most exposed to the funding squeeze you flagged in your project context.

## Step 7 — Join and Build Summary Tables
Joining charity base data with financials. Building two summary tables 
for Tableau: charity count and income by size band, and charity count 
by sector type. Pre-aggregating here keeps the Tableau file small and fast.

In [32]:
# Join charity base with financials
master = charity_clean.merge(
    parta_latest[[
        'organisation_number',
        'fin_period_end_date',
        'income_from_government_contracts',
        'income_from_government_grants',
        'charity_receives_govt_funding_contracts',
        'charity_receives_govt_funding_grants',
        'count_volunteers'
    ]],
    on='organisation_number',
    how='left'
)

print(f"Master dataset rows: {len(master):,}")

# Summary 1: charity count by income band
by_income_band = (
    master
    .groupby('income_band')
    .agg(
        charity_count=('organisation_number', 'count'),
        median_income=('latest_income', 'median'),
        total_income=('latest_income', 'sum'),
        receives_govt_grants=('charity_receives_govt_funding_grants', 
                              lambda x: (x == 'True').sum()),
        receives_govt_contracts=('charity_receives_govt_funding_contracts', 
                                 lambda x: (x == 'True').sum()),
    )
    .reset_index()
)

# Sort bands logically
band_order = ['Unknown','Under 10k','10k-100k','100k-500k','500k-1m','Over 1m']
by_income_band['income_band'] = pd.Categorical(
    by_income_band['income_band'], categories=band_order, ordered=True
)
by_income_band = by_income_band.sort_values('income_band')

print("\nBy income band:")
print(by_income_band.to_string())

# Summary 2: charity count by type (top 17)
by_type = (
    what_clean
    .merge(
        master[['organisation_number', 'income_band', 'latest_income']],
        on='organisation_number',
        how='left'
    )
    .groupby('classification_description')
    .agg(
        charity_count=('organisation_number', 'count'),
        median_income=('latest_income', 'median')
    )
    .sort_values('charity_count', ascending=False)
    .reset_index()
)

print("\nBy charity type:")
print(by_type.to_string())

Master dataset rows: 184,054

By income band:
  income_band  charity_count  median_income  total_income  receives_govt_grants  receives_govt_contracts
5     Unknown          22468            NaN  0.000000e+00                     0                        0
4   Under 10k          59652         2003.0  1.792021e+08                  4001                      193
1    10k-100k          59284        27864.0  2.164808e+09                 11645                      946
0   100k-500k          27553       192000.0  6.155225e+09                  7634                     1936
2     500k-1m           5883       685485.0  4.179856e+09                  2266                      889
3     Over 1m           9214      2738944.5  9.354874e+10                  4217                     2398

By charity type:
                                        classification_description  charity_count  median_income
0                                               Education/training         134378        23572.0
1      

This is excellent data. A few things immediately jump out that are analytically powerful for the project:

- Over 1m charities receive the most government contracts (2,398) — the largest charities are most embedded in government funding, making them most exposed to cuts
- Under 10k charities still receive 4,001 government grants — small charities are dependent on grants despite having almost no income
- Accommodation/housing has the highest median income (£52k) despite being a relatively small category — worth noting
- Religious Activities median income is £44k — higher than education despite being similar in count

In [33]:
# Export ONLY summary tables for Tableau — small and fast
tableau_file = os.path.join(OUT, "charity_commission_cleaned.xlsx")

with pd.ExcelWriter(tableau_file, engine='openpyxl') as writer:
    by_income_band.to_excel(
        writer, sheet_name='View5 - By Income Band', index=False)
    by_type.to_excel(
        writer, sheet_name='View5 - By Type', index=False)

print(f"Tableau file saved: {tableau_file}")

Tableau file saved: C:\Portfolio\P1-AI-nonprofit\data\charity_commission_cleaned.xlsx


**charity_commission_cleaned.xlsx — Tableau data source for View 5 (Who Gets Left Behind).**
Contains two pre-aggregated summary sheets: 
- *_View5 - By Income Band_* (184,054 registered UK charities grouped by income size — Under £10k through Over £1m — with charity counts, median income, and government funding dependency) and 
- *_View5 - By Type_* (charity counts and median income across 17 sector categories from Education to Housing). 
Pre-aggregated from the full 184k-row Charity Commission register so Tableau Public can connect directly without handling large raw files.

In [34]:
# Export full cleaned data as CSV files — efficient and auditable
master.to_csv(
    os.path.join(OUT, "raw_charities_clean.csv"), index=False)
what_clean.to_csv(
    os.path.join(OUT, "raw_classification_clean.csv"), index=False)
area_clean.to_csv(
    os.path.join(OUT, "raw_area_operation_clean.csv"), index=False)
parta_latest.to_csv(
    os.path.join(OUT, "raw_financials_clean.csv"), index=False)

print("CSV files saved.")

CSV files saved.


Four cleaned CSV files saved to /data/ — audit trail and reference data, not used directly in Tableau.

- raw_charities_clean.csv — 184,054 registered charities with income, expenditure, postcode, and income band. The master dataset everything else joins to.
- raw_classification_clean.csv — charity type classifications (What/Who/How) for all charities. Source of the 17 sector categories used in the Tableau summary.
- raw_area_operation_clean.csv — geographic reach per charity (Local Authority, Region, Country level). Available for deeper geographic analysis if needed later.
- raw_financials_clean.csv — latest financial year per charity including government contract income, government grant income, and volunteer counts.

These CSVs go on GitHub in /data/ as evidence that your Tableau summary tables are derived from real, traceable open government data. They also mean you can re-run any analysis later without reprocessing the original 150MB+ raw TSV files.

In [35]:
# Preview the output Excel file
preview = pd.read_excel(
    os.path.join(OUT, "charity_commission_cleaned.xlsx"),
    sheet_name=None  # loads all sheets
)

for sheet_name, df in preview.items():
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet_name}  ({len(df):,} rows x {len(df.columns)} cols)")
    print(f"{'='*60}")
    print(df.head())


Sheet: View5 - By Income Band  (6 rows x 6 cols)
  income_band  charity_count  median_income  total_income  \
0     Unknown          22468            NaN             0   
1   Under 10k          59652         2003.0     179202085   
2    10k-100k          59284        27864.0    2164807714   
3   100k-500k          27553       192000.0    6155224800   
4     500k-1m           5883       685485.0    4179856371   

   receives_govt_grants  receives_govt_contracts  
0                     0                        0  
1                  4001                      193  
2                 11645                      946  
3                  7634                     1936  
4                  2266                      889  

Sheet: View5 - By Type  (17 rows x 3 cols)
                     classification_description  charity_count  median_income
0                            Education/training         134378        23572.0
1                   General Charitable Purposes          87895        18134.0

Quick sanity check — previews the first 5 rows of every sheet in charity_commission_cleaned.xlsx to confirm the export worked correctly before connecting Tableau. Verifies sheet names, row counts, column structure, and that data looks as expected. Run once after saving the file — not needed again after that.

## Step 8 — Government Funding Dependency
Calculating what proportion of income comes from government sources 
(contracts + grants) by income band. Key finding: government contracts 
flow disproportionately to larger charities despite small charities 
being more numerous. Important equity finding for View 5.

**Data note:** Mean government income % for Under £10k band shows 245% — 
a data quality artefact where government grants exceed self-reported total 
income for some very small charities. Use median figures in Tableau, not mean.

In [36]:
# Government funding dependency by income band
# Note: govt funding columns already in master from earlier merge

# Total government income
master['total_govt_income'] = (
    master['income_from_government_contracts'].fillna(0) +
    master['income_from_government_grants'].fillna(0)
)

# Government income as % of total income
master['govt_income_pct'] = (
    master['total_govt_income'] /
    master['latest_income'].replace(0, pd.NA)
) * 100

# Summary by income band
govt_by_band = (
    master
    .groupby('income_band')
    .agg(
        charity_count=('organisation_number', 'count'),
        median_govt_income_pct=('govt_income_pct', 'median'),
        mean_govt_income_pct=('govt_income_pct', 'mean'),
        charities_receiving_grants=('charity_receives_govt_funding_grants',
                                    lambda x: (x == 'True').sum()),
        charities_receiving_contracts=('charity_receives_govt_funding_contracts',
                                       lambda x: (x == 'True').sum()),
    )
    .reset_index()
)

# Sort bands logically
govt_by_band['income_band'] = pd.Categorical(
    govt_by_band['income_band'], categories=band_order, ordered=True
)
govt_by_band = govt_by_band.sort_values('income_band')

print("Government funding dependency by income band:")
print(govt_by_band.to_string())

Government funding dependency by income band:
  income_band  charity_count median_govt_income_pct mean_govt_income_pct  charities_receiving_grants  charities_receiving_contracts
5     Unknown          22468                    NaN                  NaN                           0                              0
4   Under 10k          59652                    0.0           245.138721                        4001                            193
1    10k-100k          59284                    0.0            14.390866                       11645                            946
0   100k-500k          27553                 0.3619            19.673309                        7634                           1936
2     500k-1m           5883               1.655986             18.97865                        2266                            889
3     Over 1m           9214               2.408247            20.685643                        4217                           2398


What this tells me — and why it matters for the project:
- The median government income percentage is near zero across all bands, meaning most charities receive little or no government funding. However the mean is significantly higher — particularly the anomalous 245% figure for Under £10k charities, which is caused by a small number of tiny charities where government grants exceed their reported total income (likely a data quality issue with self-reported income figures rather than a real phenomenon).
- The more analytically meaningful finding is in the count columns. Larger charities dominate government contracts — Over £1m charities hold 2,398 contracts versus just 193 for Under £10k charities, despite there being six times more small charities. This is your equity argument: government contract income flows overwhelmingly to large organisations, while small charities serving marginalised communities rely on grants (4,001 receiving grants in the Under £10k band) which are more precarious and harder to sustain.
- For the dashboard View 5: use charities_receiving_contracts and charities_receiving_grants as a grouped bar chart by income band. The contrast between small charities' grant dependency and large charities' contract income tells a clear story about structural inequality in how public money flows through the sector — directly relevant to which organisations can afford to invest in AI capacity.

Note the 22,468 Unknown band — these charities have no income data at all, likely very small or dormant organisations not required to file full returns. Exclude them from percentage calculations but keep them visible as a data gap finding.

## Step 9 — Regional Summary
Grouping charities by UK metropolitan region using parent area from 
Local Authority level data.

**Data limitation:** Only 7 metropolitan areas returned. County-level 
charities (Kent, Surrey, Hampshire etc) do not have a parent region 
assigned in this dataset. This is a structural limitation of the 
Charity Commission register geographic hierarchy — flagged as a data 
gap in the dashboard.

In [37]:
# Region level only has 4 broad areas — use parent_geographic_area_description
# from Local Authority rows to get proper UK regions

local_auth = area_clean[
    area_clean['geographic_area_type'] == 'Local Authority'
].copy()

# parent_geographic_area_description gives us the actual UK regions
region_summary = (
    local_auth
    .merge(
        master[[
            'organisation_number',
            'income_band',
            'latest_income',
            'latest_expenditure',
            'charity_receives_govt_funding_grants',
            'charity_receives_govt_funding_contracts'
        ]],
        on='organisation_number',
        how='left'
    )
)

# Summary by parent region
by_region = (
    region_summary
    .groupby('parent_geographic_area_description')
    .agg(
        charity_count=('organisation_number', 'count'),
        median_income=('latest_income', 'median'),
        total_income=('latest_income', 'sum'),
        charities_receiving_grants=('charity_receives_govt_funding_grants',
                                    lambda x: (x == 'True').sum()),
        charities_receiving_contracts=('charity_receives_govt_funding_contracts',
                                       lambda x: (x == 'True').sum()),
    )
    .reset_index()
    .rename(columns={'parent_geographic_area_description': 'region'})
    .sort_values('charity_count', ascending=False)
)

print(f"Regions found: {len(by_region)}")
print("\nBy region:")
print(by_region.to_string())

Regions found: 7

By region:
               region  charity_count  median_income  total_income  charities_receiving_grants  charities_receiving_contracts
0      Greater London          31296        45513.0  1.503646e+10                        3609                           1246
5       West Midlands          14522        34743.0  4.759563e+09                        1449                            529
1  Greater Manchester          14380        39644.0  5.300289e+09                        1769                            615
6      West Yorkshire           9669        31348.0  3.011409e+09                        1271                            384
2          Merseyside           7662        29119.0  2.576709e+09                         833                            305
4       Tyne And Wear           7237        46725.0  2.477499e+09                         961                            326
3     South Yorkshire           5807        34495.0  1.929580e+09                         781   

Only 7 regions returned because parent_geographic_area_description at Local Authority level only captures metropolitan counties and combined authorities — not the full set of English regions like South East, North West, East of England etc. The non-metropolitan areas (counties like Kent, Surrey, Hampshire) don't have a parent region assigned in this field, so they're being dropped.This is a data structure limitation in the Charity Commission register — the geographic hierarchy is inconsistent. Metropolitan areas have a parent region, rural and county areas don't.

**2 options**

- Option A: se what you have and be honest about it
Keep the 7 metropolitan regions as a partial picture. Label it clearly in Tableau as "Major metropolitan areas only — county-level charities excluded due to data structure." This is actually a valid data gap finding for your dashboard.
- Option B: Use Local Authority directly
Skip the region grouping and use Local Authority level instead — you'll get granular data for every area but it will be 300+ rows which is harder to visualise cleanly in Tableau.

Rcommendation: Option A.

The 7 metropolitan areas you have are analytically interesting on their own — Greater London dominates with 31,296 charities and the highest median income (£45,513), while Merseyside and South Yorkshire have significantly lower median incomes despite sizeable charity counts. That inequality story is exactly what View 5 needs.

For your dashboard: present this as a metropolitan area comparison chart, add a data gap note explaining county-level charities aren't captured, and flag this explicitly as a limitation of the open dataset. That's more credible than pretending you have complete regional coverage.

## Step 10 — Export Tableau Data Source
Exporting four pre-aggregated summary sheets to a single Excel file 
for Tableau Public. Pre-aggregating avoids loading 184,000+ rows 
into Tableau — only the summary counts and medians are needed for 
the View 5 charts.

In [38]:
# Final export — all summary tables for Tableau
tableau_file = os.path.join(OUT, "charity_commission_cleaned.xlsx")

print(f"Saving to {tableau_file}...")

with pd.ExcelWriter(tableau_file, engine='openpyxl') as writer:
    by_income_band.to_excel(
        writer, sheet_name='View5 - By Income Band', index=False)
    by_type.to_excel(
        writer, sheet_name='View5 - By Type', index=False)
    govt_by_band.to_excel(
        writer, sheet_name='View5 - Govt Funding', index=False)
    by_region.to_excel(
        writer, sheet_name='View5 - By Region', index=False)

print("Done.")
print("\nSheets written:")
print("  View5 - By Income Band  — charity count and income by size band")
print("  View5 - By Type         — charity count by sector category")
print("  View5 - Govt Funding    — government funding dependency by size band")
print("  View5 - By Region       — charity count and income by metropolitan area")
print("\nNote: By Region covers 7 metropolitan areas only.")
print("County-level charities excluded — data gap flagged in dashboard.")

Saving to C:\Portfolio\P1-AI-nonprofit\data\charity_commission_cleaned.xlsx...
Done.

Sheets written:
  View5 - By Income Band  — charity count and income by size band
  View5 - By Type         — charity count by sector category
  View5 - Govt Funding    — government funding dependency by size band
  View5 - By Region       — charity count and income by metropolitan area

Note: By Region covers 7 metropolitan areas only.
County-level charities excluded — data gap flagged in dashboard.


charity_commission_cleaned.xlsx — final Tableau data source for View 5. Exports four pre-aggregated summary sheets: charity distribution by income band, by sector type, by government funding dependency, and by metropolitan region. This is the file you connect Tableau Public to directly for building View 5 charts.

## Step 11 — Export Audit CSVs
Saving full cleaned datasets as CSVs for GitHub. These are the complete 
underlying data behind the Tableau summary tables — not used in Tableau 
directly but stored as evidence that all summary figures are derived 
from real, traceable open government data.

In [39]:
# Export full cleaned data as CSV files — audit trail for GitHub
master.to_csv(
    os.path.join(OUT, "raw_charities_clean.csv"), index=False)
what_clean.to_csv(
    os.path.join(OUT, "raw_classification_clean.csv"), index=False)
area_clean.to_csv(
    os.path.join(OUT, "raw_area_operation_clean.csv"), index=False)
parta_latest.to_csv(
    os.path.join(OUT, "raw_financials_clean.csv"), index=False)

print("CSV audit files saved:")
print("  raw_charities_clean.csv")
print("  raw_classification_clean.csv")
print("  raw_area_operation_clean.csv")
print("  raw_financials_clean.csv")

CSV audit files saved:
  raw_charities_clean.csv
  raw_classification_clean.csv
  raw_area_operation_clean.csv
  raw_financials_clean.csv


Four cleaned CSV files saved to /data/ — full underlying datasets for audit and GitHub. These are the complete cleaned versions of each Charity Commission table before aggregation. Not used in Tableau — they exist as your evidence trail proving the summary tables are derived from real open government data, and as a reference if you need to re-run or extend the analysis later.

In [40]:
# Data quality summary — final check
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\nSource files processed:        4")
print(f"Registered charities:          {len(charity_clean):,}")
print(f"Removed / excluded:            {len(charity) - len(charity_clean):,}")
print(f"Charities with income data:    {charity_clean['latest_income'].notna().sum():,}")
print(f"Charities missing income:      {charity_clean['latest_income'].isna().sum():,}")
print(f"Charities with postcode:       {charity_clean['charity_contact_postcode'].notna().sum():,}")

print(f"\nClassification rows (What):    {len(what_clean):,}")
print(f"Unique charity types:          {what_clean['classification_description'].nunique()}")

print(f"\nArea of operation rows:        {len(area_clean):,}")
print(f"Metropolitan regions captured: {len(by_region)}")

print(f"\nFinancial records (latest yr): {len(parta_latest):,}")
print(f"Receiving govt grants:         {(parta_latest['charity_receives_govt_funding_grants'] == 'True').sum():,}")
print(f"Receiving govt contracts:      {(parta_latest['charity_receives_govt_funding_contracts'] == 'True').sum():,}")

print(f"\nTableau output sheets:         4")
print(f"Audit CSV files:               4")
print(f"\nData gap flagged:              County-level regional data incomplete")
print(f"Data gap flagged:              No AI adoption data in register")
print(f"Data gap flagged:              Under 10k income band has anomalous mean govt %")
print("\n" + "=" * 60)
print("Notebook complete. Ready to connect Tableau Public.")
print("=" * 60)

DATA QUALITY SUMMARY

Source files processed:        4
Registered charities:          184,054
Removed / excluded:            210,565
Charities with income data:    161,586
Charities missing income:      22,468
Charities with postcode:       167,909

Classification rows (What):    644,414
Unique charity types:          17

Area of operation rows:        537,048
Metropolitan regions captured: 7

Financial records (latest yr): 174,263
Receiving govt grants:         32,990
Receiving govt contracts:      7,360

Tableau output sheets:         4
Audit CSV files:               4

Data gap flagged:              County-level regional data incomplete
Data gap flagged:              No AI adoption data in register
Data gap flagged:              Under 10k income band has anomalous mean govt %

Notebook complete. Ready to connect Tableau Public.
